# EarGate — Обучение на синтетике + sireNNet

Обучает акустический классификатор сирен для распознавания транспортных средств экстренных служб
(пожарная машина / скорая помощь / полиция) на:
1. Синтетические данные с эффектом Доплера, реверберацией, искажением и дорожным шумом
2. Реальные записи из sireNNet (Mendeley Data, CC BY)

Результат: `siren_classifier.pkl` + `siren_scaler.pkl` для развертывания на Raspberry Pi.

**Необходимый набор данных**: `sirennet` (загружен как Dataset Kaggle из Mendeley).

In [ ]:
import sys
import os
import json
import logging
import zipfile
import shutil
import warnings
from pathlib import Path

import numpy as np
import librosa
import soundfile as sf
import joblib
from scipy.signal import resample_poly
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("eargate_notebook")

# Куда сохранять выходные файлы
WORK_DIR = "/kaggle/working/eargate"
OUTPUT_DIR = "/kaggle/working/eargate_output"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Librosa: {librosa.__version__}")
print(f"Рабочий каталог: {WORK_DIR}")

In [ ]:
%%writefile siren_config.py
"""Все константы EarGate. Переопределение через env."""
import os

# Audio
SAMPLE_RATE = 22050
CHUNK_DURATION = 3.0
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION)
HOP_DURATION = 0.5
HOP_SAMPLES = int(SAMPLE_RATE * HOP_DURATION)

# Энергетический вентиль
ENERGY_THRESHOLD = 0.01
SIREN_FREQ_MIN = 150
SIREN_FREQ_MAX = 2000

# MFCC
N_MFCC = 13
N_FFT = 2048
HOP_LENGTH_MFCC = 512
N_MELS = 128
N_FEATURES = 48

# Классификатор
MODEL_PATH = "siren_classifier.pkl"
SCALER_PATH = "siren_scaler.pkl"
CONFIDENCE_THRESHOLD = 0.65

# Защита от ложных срабатываний
CONSECUTIVE_HITS = 2
PROBABILITY_HISTORY_LEN = 3
MIN_SIREN_DURATION = 0.8
COOLDOWN_SECONDS = 30

# Датчик / реле
SENSOR_RESET_DELAY = 5
RELAY_PULSE_SECONDS = 2.0

# Журналы
DETECTION_LOG_DIR = "logs/detections"
MAX_LOG_AGE_DAYS = 7
MAX_LOG_SIZE_MB = 500

# Classes
SIREN_CLASSES = {
    0: "fire_truck",
    1: "ambulance",
    2: "police_car",
    3: "none",
}
SIREN_LABELS_RU = {
    "fire_truck": "пожарная машина",
    "ambulance": "скорая помощь",
    "police_car": "полиция",
    "none": "—",
}

# Источник звука (не используется при обучении)
AUDIO_SOURCE = "microphone"
RTSP_AUDIO_URL_ENTRY = ""
RTSP_AUDIO_URL_EXIT = ""
MIC_ENTRY_DEVICE_INDEX = 0
MIC_EXIT_DEVICE_INDEX = 1

# Яндекс (заглушки для обучения)
SKILL_ID = ""
OAUTH_TOKEN = ""
USER_ID = None
DEVICES = {
    "fire_truck_entry": {"id": "sensor-fire-entry", "name": "Пожарная — въезд"},
    "ambulance_entry": {"id": "sensor-ambulance-entry", "name": "Скорая — въезд"},
    "police_car_entry": {"id": "sensor-police-entry", "name": "Полиция — въезд"},
    "fire_truck_exit": {"id": "sensor-fire-exit", "name": "Пожарная — выезд"},
    "ambulance_exit": {"id": "sensor-ambulance-exit", "name": "Скорая — выезд"},
    "police_car_exit": {"id": "sensor-police-exit", "name": "Полиция — выезд"},
}
BARRIER_RELAY_DEVICE_ID = "relay-barrier"
YANDEX_NOTIFY_URL = ""
WEBHOOK_PORT = 8080

In [ ]:
%%writefile siren_features.py
"""Извлечение 48 признаков из 3-секундного аудиофрагмента. Одиночный STFT, досрочный выход."""
import numpy as np
import librosa

from siren_config import (
    SAMPLE_RATE, N_MFCC, N_FFT, HOP_LENGTH_MFCC, N_MELS,
    SIREN_FREQ_MIN, SIREN_FREQ_MAX, N_FEATURES
)


def extract_features(audio, sr=SAMPLE_RATE):
    audio = audio.astype(np.float32)
    abs_rms = float(np.sqrt(np.mean(audio ** 2)))

    norm_audio = audio.copy()
    peak = np.max(np.abs(norm_audio))
    if peak > 1e-10:
        norm_audio = norm_audio / peak

    mag = np.abs(librosa.stft(norm_audio, n_fft=N_FFT,
                              hop_length=HOP_LENGTH_MFCC))
    power = mag ** 2
    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)

    siren_mask = (freqs >= SIREN_FREQ_MIN) & (freqs <= SIREN_FREQ_MAX)
    siren_energy = np.mean(mag[siren_mask, :]) if siren_mask.any() else 0.0
    total_energy = float(np.mean(mag)) + 1e-10
    siren_energy_ratio = float(siren_energy / total_energy)
    in_siren_band = siren_energy_ratio > 0.3

    if not in_siren_band:
        return np.zeros(N_FEATURES, dtype=np.float32), False

    mel_S = librosa.feature.melspectrogram(S=power, sr=sr, n_mels=N_MELS)
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel_S), n_mfcc=N_MFCC)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)
    mfcc_delta = librosa.feature.delta(mfcc)
    delta_mean = np.mean(mfcc_delta, axis=1)

    centroid = librosa.feature.spectral_centroid(S=mag, sr=sr)
    centroid_mean = float(np.mean(centroid))
    centroid_std = float(np.std(centroid))
    modulation_rate = float(np.mean(np.abs(np.diff(centroid[0]))))
    centroid_ac = _autocorr_periodicity(centroid[0])

    zcr = librosa.feature.zero_crossing_rate(norm_audio,
                                             frame_length=N_FFT,
                                             hop_length=HOP_LENGTH_MFCC)
    zcr_mean = float(np.mean(zcr))

    flatness = librosa.feature.spectral_flatness(S=mag)
    flatness_mean = float(np.mean(flatness))

    rolloff = librosa.feature.spectral_rolloff(S=mag, sr=sr,
                                               roll_percent=0.85)
    rolloff_mean = float(np.mean(rolloff))

    features = np.concatenate([
        mfcc_mean, mfcc_std, delta_mean,
        [centroid_mean, centroid_std],
        [zcr_mean], [abs_rms],
        [flatness_mean], [rolloff_mean],
        [modulation_rate], [siren_energy_ratio],
        [centroid_ac],
    ]).astype(np.float32)

    if features.shape[0] != N_FEATURES:
        raise ValueError(f"Expected {N_FEATURES}, got {features.shape[0]}")

    return features, True


def _autocorr_periodicity(signal):
    if len(signal) < 4:
        return 0.0
    sig = signal - np.mean(signal)
    ac = np.correlate(sig, sig, mode="full")
    ac = ac[len(ac) // 2:]
    if ac[0] < 1e-10:
        return 0.0
    ac_norm = ac / ac[0]
    for i in range(2, min(len(ac_norm) - 1, 50)):
        if (ac_norm[i] > 0.3 and ac_norm[i] > ac_norm[i - 1]
                and ac_norm[i] > ac_norm[i + 1]):
            return float(ac_norm[i])
    return 0.0

In [ ]:
%%writefile siren_classifier.py
import logging
import joblib
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, accuracy_score

from siren_config import (
    MODEL_PATH, SCALER_PATH, SIREN_CLASSES,
    CONFIDENCE_THRESHOLD, N_FEATURES
)
from siren_features import extract_features

logger = logging.getLogger("classifier")


class SirenClassifier:
    def __init__(self):
        self.model = None
        self.scaler = None

    def load(self):
        self.model = joblib.load(MODEL_PATH)
        self.scaler = joblib.load(SCALER_PATH)
        logger.info("Model loaded (%d features)", N_FEATURES)

    def save(self):
        joblib.dump(self.model, MODEL_PATH)
        joblib.dump(self.scaler, SCALER_PATH)

    def predict_proba(self, audio, sr):
        features, in_band = extract_features(audio, sr)
        if not in_band:
            return np.array([0.0, 0.0, 0.0, 1.0], dtype=np.float32)
        fs = self.scaler.transform(features.reshape(1, -1))
        return self.model.predict_proba(fs)[0]

    def predict(self, audio, sr):
        proba = self.predict_proba(audio, sr)
        i = int(np.argmax(proba))
        cls = SIREN_CLASSES[i]
        conf = float(proba[i])
        if conf < CONFIDENCE_THRESHOLD:
            return "none", conf
        return cls, conf

    def train(self, X, y):
        if X.shape[1] != N_FEATURES:
            raise ValueError(f"X has {X.shape[1]} features, expected {N_FEATURES}")

        min_class_size = int(np.bincount(y).min())
        logger.info("Min class size: %d", min_class_size)

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y,
        )

        self.scaler = StandardScaler()
        X_train_s = self.scaler.fit_transform(X_train)
        X_test_s = self.scaler.transform(X_test)

        base = SVC(kernel="rbf", C=10, gamma="scale", random_state=42)
        if min_class_size >= 3:
            self.model = CalibratedClassifierCV(base, cv=3)
        elif min_class_size == 2:
            self.model = CalibratedClassifierCV(base, cv=2)
        else:
            logger.warning("Class with 1 sample — calibration disabled")
            self.model = SVC(kernel="rbf", C=10, gamma="scale",
                             probability=True, random_state=42)

        self.model.fit(X_train_s, y_train)

        y_pred = self.model.predict(X_test_s)
        acc = accuracy_score(y_test, y_pred)
        labels = sorted(set(y))
        target_names = [SIREN_CLASSES[i] for i in labels]
        logger.info("Accuracy: %.4f", acc)
        logger.info("\n%s", classification_report(
            y_test, y_pred, target_names=target_names, zero_division=0))

        self.save()
        return acc

In [ ]:
%%writefile siren_synthesizer.py
"""Синтетический набор данных: доплер, реверберация, дисторсия, дорожный шум."""
import numpy as np
from siren_config import SAMPLE_RATE, CHUNK_DURATION
from siren_features import extract_features

SAMPLES_PER_CHUNK = int(SAMPLE_RATE * CHUNK_DURATION)


def generate_wail(fmin, fmax, cycle, dur, sr=SAMPLE_RATE):
    t = np.linspace(0, dur, int(sr * dur))
    phase = (t % cycle) / cycle
    freq = fmin + (fmax - fmin) * 0.5 * (1 - np.cos(2 * np.pi * phase))
    return np.sin(2 * np.pi * np.cumsum(freq) / sr)


def generate_yelp(fmin, fmax, cycle, dur, sr=SAMPLE_RATE):
    t = np.linspace(0, dur, int(sr * dur))
    phase = (t % cycle) / cycle
    freq = fmin + (fmax - fmin) * phase
    return np.sin(2 * np.pi * np.cumsum(freq) / sr)


def generate_hilo(f_low, f_high, switch, dur, sr=SAMPLE_RATE):
    t = np.linspace(0, dur, int(sr * dur))
    freq = np.where((t // switch) % 2 == 0, f_high, f_low)
    return np.sin(2 * np.pi * np.cumsum(freq) / sr)


def generate_airhorn(freq, dur, sr=SAMPLE_RATE):
    t = np.linspace(0, dur, int(sr * dur))
    signal = (np.sin(2 * np.pi * freq * t)
              + 0.5 * np.sin(2 * np.pi * freq * 2 * t)
              + 0.25 * np.sin(2 * np.pi * freq * 3 * t))
    env = np.minimum(t / 0.05, 1.0) * np.minimum((dur - t) / 0.1, 1.0)
    return signal * env


def add_doppler(signal, sr=SAMPLE_RATE, speed=15, min_dist=10.0):
    c = 343.0
    n = len(signal)
    t = np.arange(n) / sr
    x = speed * t
    r = np.sqrt(x ** 2 + min_dist ** 2)
    v_r = (x * speed) / r
    factor = c / (c - v_r)
    idx = np.clip(np.cumsum(factor) - 1, 0, n - 1)
    return np.interp(idx, np.arange(n), signal)


def _pink_noise(n, sr=SAMPLE_RATE):
    white = np.random.randn(n)
    f = np.fft.rfftfreq(n, 1 / sr)
    f[0] = 1
    pink = np.fft.irfft(np.fft.rfft(white) / np.sqrt(f), n=n)
    return pink / (np.std(pink) + 1e-10)


def add_traffic_noise(signal, sr=SAMPLE_RATE, snr_db=10):
    pink = _pink_noise(len(signal), sr)
    t = np.linspace(0, len(signal) / sr, len(signal))
    engine = np.sin(2 * np.pi * np.random.uniform(80, 120) * t) * 0.3
    noise = pink * 0.7 + engine * 0.3
    sig_p = np.mean(signal ** 2)
    noise_p = sig_p / (10 ** (snr_db / 10))
    noise = noise * np.sqrt(noise_p) / (np.std(noise) + 1e-10)
    return signal + noise


def add_reverb(signal, sr=SAMPLE_RATE, rt60=0.3):
    length = int(sr * rt60)
    impulse = np.zeros(length)
    impulse[0] = 1.0
    impulse *= np.exp(-np.linspace(0, 6, length))
    return np.convolve(signal, impulse, mode="full")[:len(signal)]


def add_distortion(signal, drive=1.5):
    return np.tanh(signal * drive)


def add_noise(signal, snr_db=15):
    sig_p = np.mean(signal ** 2)
    noise_p = sig_p / (10 ** (snr_db / 10))
    return signal + np.random.randn(len(signal)) * np.sqrt(noise_p)


def augment(signal, sr=SAMPLE_RATE):
    signal = add_distortion(signal, drive=np.random.uniform(1.2, 2.0))
    if np.random.random() > 0.3:
        signal = add_doppler(signal, sr, speed=np.random.uniform(5, 20))
    if np.random.random() > 0.3:
        signal = add_reverb(signal, sr, rt60=np.random.uniform(0.1, 0.5))
    signal = add_traffic_noise(signal, sr, snr_db=np.random.uniform(3, 20))
    signal = add_noise(signal, snr_db=np.random.uniform(5, 25))
    peak = np.max(np.abs(signal))
    if peak > 1.0:
        signal = signal / peak
    return signal.astype(np.float32)


def generate_dataset(n_samples_per_class=300):
    X, y = [], []
    dur = CHUNK_DURATION

    for _ in range(n_samples_per_class):
        # Fire truck
        signal = generate_wail(np.random.uniform(150, 300),
                               np.random.uniform(600, 1000),
                               np.random.uniform(2.5, 4.0), dur)
        if np.random.random() > 0.5:
            airhorn = generate_airhorn(np.random.uniform(200, 400),
                                       np.random.uniform(0.5, 1.5))
            off = np.random.randint(0, len(signal) - len(airhorn))
            signal[off:off + len(airhorn)] += airhorn
        signal = augment(signal)
        feats, _ = extract_features(signal)
        X.append(feats); y.append(0)

        # Ambulance
        signal = generate_hilo(np.random.uniform(500, 700),
                               np.random.uniform(1000, 1500),
                               np.random.uniform(0.3, 0.7), dur)
        if np.random.random() > 0.6:
            wail = generate_wail(np.random.uniform(400, 600),
                                 np.random.uniform(800, 1200),
                                 np.random.uniform(1.5, 2.5), dur)
            signal = 0.5 * signal + 0.5 * wail
        signal = augment(signal)
        feats, _ = extract_features(signal)
        X.append(feats); y.append(1)

        # Police
        signal = generate_yelp(np.random.uniform(500, 800),
                               np.random.uniform(1200, 2000),
                               np.random.uniform(0.3, 0.8), dur)
        if np.random.random() > 0.5:
            phaser = generate_yelp(np.random.uniform(800, 1200),
                                   np.random.uniform(1500, 2000),
                                   np.random.uniform(0.1, 0.3), dur)
            signal = 0.6 * signal + 0.4 * phaser
        signal = augment(signal)
        feats, _ = extract_features(signal)
        X.append(feats); y.append(2)

        # None
        snr = np.random.choice([0.001, 0.005, 0.01, 0.02])
        signal = np.random.randn(SAMPLES_PER_CHUNK).astype(np.float32) * snr
        if np.random.random() > 0.7:
            airhorn = generate_airhorn(np.random.uniform(1800, 3550),
                                       np.random.uniform(0.2, 0.5))
            off = np.random.randint(0, len(signal) - len(airhorn))
            signal[off:off + len(airhorn)] += airhorn * 0.3
        signal = augment(signal)
        feats, _ = extract_features(signal)
        X.append(feats); y.append(3)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

In [ ]:
import importlib
import siren_config, siren_features, siren_classifier, siren_synthesizer
for m in (siren_config, siren_features, siren_classifier, siren_synthesizer):
    importlib.reload(m)

from siren_config import SIREN_CLASSES, N_FEATURES, SAMPLE_RATE, CHUNK_SAMPLES
from siren_features import extract_features
from siren_classifier import SirenClassifier
from siren_synthesizer import generate_dataset

print(f"Признаков: {N_FEATURES}")
print(f"Классы: {SIREN_CLASSES}")
print(f"Частота дискретизации: {SAMPLE_RATE}")
print("✓ Модули готовы")

In [ ]:
print("Генерация синтетического набора данных (300 на класс)...")
X_syn, y_syn = generate_dataset(n_samples_per_class=300)
print(f"Синтетика: {X_syn.shape[0]} образцов, {X_syn.shape[1]} признаков")
print(f"Распределение: {np.bincount(y_syn)}")

np.save("X_syn.npy", X_syn)
np.save("y_syn.npy", y_syn)
print("✓ Сохранено в X_syn.npy, y_syn.npy")

In [ ]:
# Автоматическое определение пути к sireNNet в /kaggle/input/
SIRENNET_PATH = None
for candidate in [
    "/kaggle/input/sirennet",
    "/kaggle/input/sirennet-dataset",
    "/kaggle/input/sire-nnet",
]:
    if os.path.isdir(candidate):
        SIRENNET_PATH = candidate
        break

# Резервный вариант: поиск каталога, содержащего подпапки классов
if SIRENNET_PATH is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs_lower = [d.lower() for d in dirs]
        if "ambulance" in dirs_lower and "firetruck" in dirs_lower:
            SIRENNET_PATH = root
            break

print(f"Путь sireNNet: {SIRENNET_PATH}")

if SIRENNET_PATH:
    # Показать структуру
    for d in sorted(os.listdir(SIRENNET_PATH)):
        full = os.path.join(SIRENNET_PATH, d)
        if os.path.isdir(full):
            wavs = [f for f in os.listdir(full) if f.lower().endswith(".wav")]
            print(f"  {d}/: {len(wavs)} wav-файлов")
else:
    print("⚠ sireNNet не найден — будем обучать только на синтетических данных")

In [ ]:
SIRENNET_LABEL_MAP = {
    "ambulance": 1,
    "firetruck": 0,
    "police": 2,
    "traffic": 3,
}


# Учитываем различия в регистре названий папок
def find_subdir(base, name):
    for d in os.listdir(base):
        if d.lower() == name.lower():
            return os.path.join(base, d)
    return None


def load_and_resample(path, target_sr=SAMPLE_RATE):
    audio, sr = librosa.load(path, sr=None, mono=True)
    if sr != target_sr:
        # Ресемплинг многофазным фильтром (быстро, высокое качество)
        from math import gcd
        g = gcd(sr, target_sr)
        audio = resample_poly(audio, target_sr // g, sr // g)
    return audio.astype(np.float32)


def load_sirennet_features(base_path):
    X_list, y_list = [], []
    skipped = 0
    if base_path is None:
        return np.array([]), np.array([])

    for label_name, label_idx in SIRENNET_LABEL_MAP.items():
        dir_path = find_subdir(base_path, label_name)
        if dir_path is None:
            logger.warning("Папка не найдена: %s", label_name)
            continue

        files = sorted([f for f in os.listdir(dir_path)
                        if f.lower().endswith(".wav")])
        logger.info("%s: %d files", label_name, len(files))

        for f in files:
            try:
                audio = load_and_resample(os.path.join(dir_path, f))
                # Приводим к CHUNK_SAMPLES
                if len(audio) < CHUNK_SAMPLES:
                    audio = np.pad(audio, (0, CHUNK_SAMPLES - len(audio)))
                audio = audio[:CHUNK_SAMPLES]

                feats, in_band = extract_features(audio)
                # Отбрасываем сэмплы класса сирены без энергии в полосе (несоответствие)
                if not in_band and label_idx != 3:
                    skipped += 1
                    continue
                X_list.append(feats)
                y_list.append(label_idx)
            except Exception as e:
                logger.warning("Пропуск %s: %s", f, e)
                skipped += 1

    logger.info("Загружено: %d, пропущено: %d", len(X_list), skipped)
    if not X_list:
        return np.array([]), np.array([])
    return (np.array(X_list, dtype=np.float32),
            np.array(y_list, dtype=np.int32))


X_real, y_real = load_sirennet_features(SIRENNET_PATH)
if X_real.size > 0:
    print(f"sireNNet: {X_real.shape[0]} образцов")
    print(f"Распределение: {np.bincount(y_real)}")
    np.save("X_real.npy", X_real)
    np.save("y_real.npy", y_real)
else:
    print("Данные sireNNet не загружены")

In [ ]:
# Объединяем: синтетика + 3× повторенные реальные + 2× дополнительные синтетические негативы
if X_real.size > 0:
    X_real_rep = np.tile(X_real, (3, 1))
    y_real_rep = np.tile(y_real, 3)

    X_syn_none = X_syn[y_syn == 3]
    y_syn_none = y_syn[y_syn == 3]
    X_syn_none_rep = np.tile(X_syn_none, (2, 1))
    y_syn_none_rep = np.tile(y_syn_none, 2)

    X = np.vstack([X_syn, X_real_rep, X_syn_none_rep])
    y = np.concatenate([y_syn, y_real_rep, y_syn_none_rep])

    print(f"Синтетика: {X_syn.shape[0]}")
    print(f"sireNNet × 3: {X_real_rep.shape[0]}")
    print(f"Синтетические негативы × 2: {X_syn_none_rep.shape[0]}")
else:
    X, y = X_syn, y_syn

print(f"\nОбъединённый датасет: {X.shape[0]} образцов, {X.shape[1]} признаков")
print(f"Распределение классов: {np.bincount(y)}")

clf = SirenClassifier()
accuracy = clf.train(X, y)
print(f"\n✓ Модель обучена, точность на тесте: {accuracy:.4f}")

In [ ]:
clf_loaded = SirenClassifier()
clf_loaded.load()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

X_test_s = clf_loaded.scaler.transform(X_test)
y_pred = clf_loaded.model.predict(X_test_s)

labels = sorted(set(y))
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=[SIREN_CLASSES[i] for i in labels],
    yticklabels=[SIREN_CLASSES[i] for i in labels],
)
plt.title("Confusion Matrix — EarGate (синтетика + sireNNet)")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print(classification_report(
    y_test, y_pred,
    target_names=[SIREN_CLASSES[i] for i in labels],
    zero_division=0,
))

In [ ]:
# Копируем артефакты модели в выходную папку
shutil.copy("siren_classifier.pkl", os.path.join(OUTPUT_DIR, "siren_classifier.pkl"))
shutil.copy("siren_scaler.pkl", os.path.join(OUTPUT_DIR, "siren_scaler.pkl"))
shutil.copy("confusion_matrix.png", os.path.join(OUTPUT_DIR, "confusion_matrix.png"))

# Отчёт об обучении
report = {
    "samples_synthetic": int(X_syn.shape[0]),
    "samples_real": int(X_real.shape[0]) if X_real.size else 0,
    "samples_combined": int(X.shape[0]),
    "n_features": int(N_FEATURES),
    "accuracy_test": float(accuracy),
    "class_distribution": np.bincount(y).tolist(),
    "siren_classes": SIREN_CLASSES,
}
with open(os.path.join(OUTPUT_DIR, "training_report.json"), "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# Zip для скачивания
shutil.make_archive(
    "/kaggle/working/eargate_model", "zip", OUTPUT_DIR,
)

print("Выходные файлы:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")
print(f"\nАрхив: /kaggle/working/eargate_model.zip")
print(f"Точность на тесте: {accuracy:.4f}")